# Baseline Diagnostic — React Dependency-Prediction Variable Tracking

범위: `research_context_260620.md` §11–§12에 해당하는 baseline diagnostic 단계만 수행한다.
activation patching(§13 이후)은 이 노트북에 포함하지 않는다.

이 실험은 `clean_target=dep`, `corrupted_target=alt`로 설계되어 있으므로 기본적으로
**dependency inclusion 판단이 아니라 variable tracking 실험**이다. 코드 주석/출력에서
"LLM이 React 의미를 이해한다" 류의 표현은 쓰지 않고, candidate circuit /
dependency-prediction-related heads / effect-body variable tracking /
non-React-Hook callback-array control 같은 표현을 쓴다.

진행 순서 (단계적):
1. 모델 로드 (Llama 3.2 1B, Gemma 3 1B-pt)
2. target id 계산 헬퍼 검증 — 1 sample assert
3. 소수 sample dry run
4. 전체 dataset 토큰 검증 (Stage 1)
5. 전체 baseline logit 측정 (Stage 2) → `baseline_logits_3tokens.csv`
6. §12 기준 평가 요약표 → `baseline_summary.md`


In [1]:
import json
from pathlib import Path

import pandas as pd
import torch

torch.set_grad_enabled(False)

from transformer_lens.model_bridge import TransformerBridge

## Configuration

In [2]:
DATASET_PATH = Path("../dataset/dataset.json")
CONDITIONS = ["useEffect", "useLayoutEffect", "alias", "alias_ctrl", "subscribe"]

MODELS = {
    "llama3.2-1b": "meta-llama/Llama-3.2-1B",
    "gemma3-1b-pt": "google/gemma-3-1b-pt",
}

with open(DATASET_PATH) as f:
    dataset = json.load(f)

print(f"Dataset: {len(dataset)} samples x {len(CONDITIONS)} conditions")

Dataset: 45 samples x 5 conditions


## Helpers

`get_target_id`: target token id를 항상 `prefix + suffix`를 실제로 토큰화해서, prefix 대비
정확히 `expected_add`개 토큰만 늘어났는지 assert로 확인한 뒤 계산한다. 문자열 형태를
하드코딩하지 않는다.

### close_id 계산 — §10.3 대비 변경점

§10.3 원안은 `prefix + "]"`가 prefix 대비 +1 토큰이라고 가정한다. 그런데 prefix가 ` [`로
끝나기 때문에 `]`를 바로 이어붙이면 Llama 3.2 1B와 Gemma 3 1B-pt 토크나이저 모두
` [` + `]`를 ` []`라는 **단일 병합 토큰**으로 재토큰화해 버린다 (두 토크나이저 모두
`"[]"`가 단일 vocab 항목으로 존재). 이 때문에 원안의 assert가 항상 실패한다.

대안: `prefix + dep + "]"` (예: `...}, [page]`)로 닫힌 1-원소 배열 문맥을 만들어 +2 토큰
assert로 검증하고, 그 마지막 토큰을 `close_id`로 취한다. 식별자가 대괄호 사이에 있을 때는
`]`가 항상 독립 토큰으로 분리되며 (`"[dep]"` → `['[', 'dep', ']']`), 이 id가 standalone
`"]"` 토큰화 결과와 정확히 일치함을 사전에 확인했다. "하드코딩하지 말고 문맥에서 계산"하는
원칙은 유지하면서 빈 대괄호 병합 아티팩트만 회피한 것이다. (연구자 확인 후 채택)

In [3]:
def get_target_id(model, prefix, prefix_n, suffix, expected_add):
    """Token id from this sample's own prefix context — never hardcoded."""
    full_tokens = model.to_tokens(prefix + suffix)
    n = full_tokens.shape[-1]
    assert n == prefix_n + expected_add, (
        f"token count mismatch: prefix_n={prefix_n} suffix={suffix!r} "
        f"got n={n} expected={prefix_n + expected_add}"
    )
    return full_tokens[0, -1].item()


def validate_and_get_ids(model, prefix, dep_str, alt_str):
    """Returns (prefix_n, dep_id, alt_id, close_id); raises AssertionError on mismatch."""
    prefix_tokens = model.to_tokens(prefix)
    prefix_n = prefix_tokens.shape[-1]

    str_tokens = model.to_str_tokens(prefix)
    assert str_tokens[-1].strip() == "[", f"last prefix token isn\'t \'[\': {str_tokens[-1]!r}"

    dep_id = get_target_id(model, prefix, prefix_n, dep_str, 1)
    alt_id = get_target_id(model, prefix, prefix_n, alt_str, 1)
    close_id = get_target_id(model, prefix, prefix_n, dep_str + "]", 2)

    return prefix_n, dep_id, alt_id, close_id


def iter_pairs(dataset, sample_ids=None):
    for item in dataset:
        if sample_ids is not None and item["id"] not in sample_ids:
            continue
        for cond in CONDITIONS:
            for run_type in ["clean", "corrupted"]:
                yield item, cond, run_type

## Step 1 — Llama 3.2 1B 로드 및 target id 계산 검증 (1 sample)

In [4]:
print(f"Loading model: {MODELS['llama3.2-1b']}")
model_llama = TransformerBridge.boot_transformers(MODELS["llama3.2-1b"], device="cpu")
print("Loaded.")

Loading model: meta-llama/Llama-3.2-1B


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded.


In [5]:
sample0 = dataset[0]
dep_str0, alt_str0 = sample0["vars"]["dep"], sample0["vars"]["alt"]
prefix0 = sample0["conditions"]["useEffect"]["clean"]

prefix_n, dep_id, alt_id, close_id = validate_and_get_ids(model_llama, prefix0, dep_str0, alt_str0)
print(f"sample={sample0['id']} dep={dep_str0!r} alt={alt_str0!r}")
print(f"prefix_n={prefix_n} dep_id={dep_id} alt_id={alt_id} close_id={close_id}")
print("ASSERTIONS PASSED (Llama 3.2 1B, 1 sample)")

sample=sub_01 dep='page' alt='ref'
prefix_n=41 dep_id=2964 alt_id=1116 close_id=60
ASSERTIONS PASSED (Llama 3.2 1B, 1 sample)


## Step 2 — Gemma 3 1B-pt 로드 및 target id 계산 검증 (1 sample)

In [6]:
print(f"Loading model: {MODELS['gemma3-1b-pt']}")
model_gemma = TransformerBridge.boot_transformers(MODELS["gemma3-1b-pt"], device="cpu")
print("Loaded.")

Loading model: google/gemma-3-1b-pt


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loaded.


In [7]:
prefix_n, dep_id, alt_id, close_id = validate_and_get_ids(model_gemma, prefix0, dep_str0, alt_str0)
print(f"sample={sample0['id']} dep={dep_str0!r} alt={alt_str0!r}")
print(f"prefix_n={prefix_n} dep_id={dep_id} alt_id={alt_id} close_id={close_id}")
print("ASSERTIONS PASSED (Gemma 3 1B-pt, 1 sample)")

sample=sub_01 dep='page' alt='ref'
prefix_n=47 dep_id=4000 alt_id=1811 close_id=236842
ASSERTIONS PASSED (Gemma 3 1B-pt, 1 sample)


## Step 3 — 소수 sample dry run (양 모델 x 5조건, 3 sample)

In [8]:
def measure_rows(model, model_key, sample_ids=None):
    rows = []
    for item, cond, run_type in iter_pairs(dataset, sample_ids):
        prefix = item["conditions"][cond][run_type]
        dep_str, alt_str = item["vars"]["dep"], item["vars"]["alt"]

        prefix_n, dep_id, alt_id, close_id = validate_and_get_ids(model, prefix, dep_str, alt_str)

        prefix_tokens = model.to_tokens(prefix)
        logits = model(prefix_tokens)
        pos = prefix_n - 1
        pos_logits = logits[0, pos, :]

        logit_dep = pos_logits[dep_id].item()
        logit_alt = pos_logits[alt_id].item()
        logit_close = pos_logits[close_id].item()

        rows.append({
            "model": model_key, "condition": cond, "sample_id": item["id"], "run_type": run_type,
            "dep_str": dep_str, "alt_str": alt_str, "n_tokens": prefix_n,
            "dep_id": dep_id, "alt_id": alt_id, "close_id": close_id,
            "logit_dep": logit_dep, "logit_alt": logit_alt, "logit_close": logit_close,
            "LD_copy": logit_dep - logit_alt,
            "LD_inclusion": logit_dep - logit_close,
            "copy_error": logit_alt - logit_close,
        })
    return rows

In [9]:
dry_run_ids = ["sub_01", "sub_02", "sub_03"]
dry_rows_llama = measure_rows(model_llama, "llama3.2-1b", dry_run_ids)
dry_df = pd.DataFrame(dry_rows_llama)
dry_df[["condition", "sample_id", "run_type", "logit_dep", "logit_alt", "logit_close", "LD_copy", "LD_inclusion"]]

,condition,sample_id,run_type,logit_dep,logit_alt,logit_close,LD_copy,LD_inclusion
0,useEffect,sub_01,clean,26.339203,22.929392,12.303530,3.409811,14.035673
1,useEffect,sub_01,corrupted,23.708712,26.172190,12.653888,-2.463478,11.054824
2,useLayoutEffect,sub_01,clean,26.083527,22.900579,12.627420,3.182947,13.456106
3,useLayoutEffect,sub_01,corrupted,23.423578,25.911531,12.977715,-2.487953,10.445863
4,alias,sub_01,clean,25.677500,22.837267,12.010870,2.840233,13.666630
5,alias,sub_01,corrupted,24.114702,25.565407,12.116290,-1.450705,11.998412
6,alias_ctrl,sub_01,clean,24.993507,22.610737,11.492312,2.382771,13.501195
7,alias_ctrl,sub_01,corrupted,23.831022,24.735090,11.439679,-0.904068,12.391343
8,subscribe,sub_01,clean,24.854420,23.600521,11.817774,1.253899,13.036646
9,subscribe,sub_01,corrupted,24.181791,24.120539,11.611733,0.061253,12.570058


## Step 4 — 전체 dataset 토큰 검증 (Stage 1, 450 조합 x 2 모델)

In [10]:
def stage1_validate(model, sample_ids=None):
    failures = []
    n_total = 0
    for item, cond, run_type in iter_pairs(dataset, sample_ids):
        n_total += 1
        prefix = item["conditions"][cond][run_type]
        dep_str, alt_str = item["vars"]["dep"], item["vars"]["alt"]
        try:
            validate_and_get_ids(model, prefix, dep_str, alt_str)
        except AssertionError as e:
            failures.append({"sample_id": item["id"], "condition": cond, "run_type": run_type, "error": str(e)})
    return n_total - len(failures), n_total, failures

In [11]:
n_ok, n_total, failures = stage1_validate(model_llama)
print(f"Llama 3.2 1B: {n_ok}/{n_total} passed")
for f in failures:
    print("  FAIL", f)

Llama 3.2 1B: 450/450 passed


In [12]:
n_ok, n_total, failures = stage1_validate(model_gemma)
print(f"Gemma 3 1B-pt: {n_ok}/{n_total} passed")
for f in failures:
    print("  FAIL", f)

Gemma 3 1B-pt: 450/450 passed


## Step 5 — 전체 baseline logit 측정 (Stage 2, 45 sample x 5 condition x clean/corrupted x 2 model)

In [13]:
rows_llama = measure_rows(model_llama, "llama3.2-1b")
print(f"llama3.2-1b: {len(rows_llama)} rows")

llama3.2-1b: 450 rows


In [14]:
rows_gemma = measure_rows(model_gemma, "gemma3-1b-pt")
print(f"gemma3-1b-pt: {len(rows_gemma)} rows")

gemma3-1b-pt: 450 rows


In [15]:
df = pd.DataFrame(rows_llama + rows_gemma)
print(df.shape)
df.to_csv("baseline_logits_3tokens.csv", index=False)
df.head()

(900, 16)


,model,condition,sample_id,run_type,dep_str,alt_str,n_tokens,dep_id,alt_id,close_id,logit_dep,logit_alt,logit_close,LD_copy,LD_inclusion,copy_error
0,llama3.2-1b,useEffect,sub_01,clean,page,ref,41,2964,1116,60,26.339203,22.929392,12.303530,3.409811,14.035673,10.625862
1,llama3.2-1b,useEffect,sub_01,corrupted,page,ref,41,2964,1116,60,23.708712,26.172190,12.653888,-2.463478,11.054824,13.518302
2,llama3.2-1b,useLayoutEffect,sub_01,clean,page,ref,43,2964,1116,60,26.083527,22.900579,12.627420,3.182947,13.456106,10.273159
3,llama3.2-1b,useLayoutEffect,sub_01,corrupted,page,ref,43,2964,1116,60,23.423578,25.911531,12.977715,-2.487953,10.445863,12.933816
4,llama3.2-1b,alias,sub_01,clean,page,ref,48,2964,1116,60,25.677500,22.837267,12.010870,2.840233,13.666630,10.826397


## Step 6 — §12 기준 평가 (요약표 → `baseline_summary.md`)

In [16]:
wide = df.pivot_table(
    index=["model", "condition", "sample_id"],
    columns="run_type",
    values=["logit_dep", "logit_alt", "logit_close", "LD_copy", "LD_inclusion", "copy_error"],
)
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()
wide["gap_copy"] = wide["LD_copy_clean"] - wide["LD_copy_corrupted"]
wide["gap_inclusion"] = wide["LD_inclusion_clean"] - wide["LD_inclusion_corrupted"]
wide.to_csv("baseline_wide_per_sample.csv", index=False)
wide.shape

(450, 17)

In [17]:
MODEL_KEYS = ["llama3.2-1b", "gemma3-1b-pt"]

summary_rows = []
for model_key in MODEL_KEYS:
    for cond in CONDITIONS:
        sub = wide[(wide["model"] == model_key) & (wide["condition"] == cond)]
        summary_rows.append({
            "model": model_key, "condition": cond, "n": len(sub),
            "median_LD_inclusion_clean": sub["LD_inclusion_clean"].median(),
            "median_LD_inclusion_corrupted": sub["LD_inclusion_corrupted"].median(),
            "median_gap_inclusion": sub["gap_inclusion"].median(),
            "frac_gap_inclusion_pos": (sub["gap_inclusion"] > 0).mean(),
            "median_LD_copy_clean": sub["LD_copy_clean"].median(),
            "median_LD_copy_corrupted": sub["LD_copy_corrupted"].median(),
            "median_gap_copy": sub["gap_copy"].median(),
            "frac_gap_copy_pos": (sub["gap_copy"] > 0).mean(),
            "median_copy_error_corrupted": sub["copy_error_corrupted"].median(),
            "mean_copy_error_corrupted": sub["copy_error_corrupted"].mean(),
            "median_copy_error_clean": sub["copy_error_clean"].median(),
        })

summary = pd.DataFrame(summary_rows)
summary.to_csv("baseline_summary_table.csv", index=False)
summary.round(3)

,model,condition,n,median_LD_inclusion_clean,median_LD_inclusion_corrupted,median_gap_inclusion,frac_gap_inclusion_pos,median_LD_copy_clean,median_LD_copy_corrupted,median_gap_copy,frac_gap_copy_pos,median_copy_error_corrupted,mean_copy_error_corrupted,median_copy_error_clean
0,llama3.2-1b,useEffect,45,14.117,10.853,3.479,1.000,4.110,-1.883,6.305,1.000,12.696,12.631,10.030
1,llama3.2-1b,useLayoutEffect,45,13.570,10.574,3.309,1.000,3.900,-1.744,5.833,1.000,12.114,12.141,9.794
2,llama3.2-1b,alias,45,13.567,11.332,2.267,1.000,2.954,-1.438,4.607,1.000,12.723,12.714,10.638
3,llama3.2-1b,alias_ctrl,45,13.309,11.679,1.623,1.000,2.503,-0.457,3.195,1.000,12.322,12.327,11.070
4,llama3.2-1b,subscribe,45,13.234,12.189,1.186,0.978,2.075,0.248,1.815,0.978,12.092,12.053,11.328
5,gemma3-1b-pt,useEffect,45,14.563,9.802,4.560,1.000,7.564,-4.384,11.929,1.000,14.062,14.177,6.825
6,gemma3-1b-pt,useLayoutEffect,45,14.360,10.466,3.950,1.000,6.142,-3.772,10.085,1.000,14.004,14.066,8.224
7,gemma3-1b-pt,alias,45,13.079,10.699,2.041,1.000,3.703,-1.904,5.667,1.000,12.765,12.877,9.236
8,gemma3-1b-pt,alias_ctrl,45,13.024,11.701,1.182,1.000,2.897,-0.917,3.752,1.000,12.632,12.672,10.077
9,gemma3-1b-pt,subscribe,45,13.449,11.815,1.584,1.000,3.307,-1.251,4.531,1.000,12.950,12.955,10.025


§12.1 기준 1~4는 명시된 부등식이므로 통과 여부를 그대로 표시한다. 기준 5(`copy_error`가
과하게 양수로 치우치지 않음)는 문서에 고정 임계값이 없으므로 수치만 제시하고 판정하지 않는다.
**메인 metric의 최종 선택은 이 노트북이 하지 않으며 연구자가 결과를 보고 결정한다.**

In [18]:
def get_row(model_key, cond="useEffect"):
    return summary[(summary["model"] == model_key) & (summary["condition"] == cond)].iloc[0]


def passfail(cond_bool):
    return "PASS" if cond_bool else "FAIL"


l = get_row("llama3.2-1b")
g = get_row("gemma3-1b-pt")

checklist = pd.DataFrame([
    {
        "#": 1, "criterion": "median LD_inclusion_clean > 0",
        "llama": l["median_LD_inclusion_clean"], "llama_verdict": passfail(l["median_LD_inclusion_clean"] > 0),
        "gemma": g["median_LD_inclusion_clean"], "gemma_verdict": passfail(g["median_LD_inclusion_clean"] > 0),
    },
    {
        "#": 2, "criterion": "median LD_inclusion_corrupted < 0",
        "llama": l["median_LD_inclusion_corrupted"], "llama_verdict": passfail(l["median_LD_inclusion_corrupted"] < 0),
        "gemma": g["median_LD_inclusion_corrupted"], "gemma_verdict": passfail(g["median_LD_inclusion_corrupted"] < 0),
    },
    {
        "#": 3, "criterion": "median gap_inclusion > 0",
        "llama": l["median_gap_inclusion"], "llama_verdict": passfail(l["median_gap_inclusion"] > 0),
        "gemma": g["median_gap_inclusion"], "gemma_verdict": passfail(g["median_gap_inclusion"] > 0),
    },
    {
        "#": 4, "criterion": "샘플 다수에서 LD_clean > LD_corrupted",
        "llama": l["frac_gap_inclusion_pos"], "llama_verdict": passfail(l["frac_gap_inclusion_pos"] > 0.5),
        "gemma": g["frac_gap_inclusion_pos"], "gemma_verdict": passfail(g["frac_gap_inclusion_pos"] > 0.5),
    },
    {
        "#": 5, "criterion": "copy_error_corrupted가 과하게 양수로 치우치지 않음 (임계값 미정)",
        "llama": l["median_copy_error_corrupted"], "llama_verdict": "(판정 보류)",
        "gemma": g["median_copy_error_corrupted"], "gemma_verdict": "(판정 보류)",
    },
])
checklist

,#,criterion,llama,llama_verdict,gemma,gemma_verdict
0,1,median LD_inclusion_clean > 0,14.116705,PASS,14.563422,PASS
1,2,median LD_inclusion_corrupted < 0,10.853230,FAIL,9.801557,FAIL
2,3,median gap_inclusion > 0,3.479086,PASS,4.559556,PASS
3,4,샘플 다수에서 LD_clean > LD_corrupted,1.000000,PASS,1.000000,PASS
4,5,copy_error_corrupted가 과하게 양수로 치우치지 않음 (임계값 미정),12.696126,(판정 보류),14.061527,(판정 보류)


## Step 7 — `baseline_summary.md` 작성

In [19]:
MODEL_LABELS = {"llama3.2-1b": "Llama 3.2 1B", "gemma3-1b-pt": "Gemma 3 1B-pt"}


def fmt(x):
    return f"{x:.3f}"


lines = []
lines.append("# Baseline Diagnostic Summary")
lines.append("")
lines.append("범위: §11–§12 baseline diagnostic. activation patching(§13 이후)은 이 단계에 포함하지 않는다.")
lines.append("")
lines.append(
    "측정 대상: 45 sample × 5 condition (`useEffect`, `useLayoutEffect`, `alias`, `alias_ctrl`, "
    "`subscribe`) × {clean, corrupted} × 2 model (Llama 3.2 1B, Gemma 3 1B-pt). "
    "총 900개 (model, condition, sample, run_type) row."
)
lines.append("")
lines.append(
    "입력은 모두 prefix-only이며 `[`로 끝난다. 측정 위치는 prefix 마지막 토큰(`[`) 위치이고, "
    "target token id는 각 prefix 문맥에서 직접 계산했다 (하드코딩 없음)."
)
lines.append("")
lines.append("## 0. close_id 계산 방식 — §10.3 대비 변경점 (연구자 확인 완료)")
lines.append("")
lines.append(
    "§10.3 원안은 `prefix + \"]\"` 토큰화가 prefix 대비 정확히 1개 토큰만 늘어난다고 가정한다. "
    "실제로는 prefix가 ` [`로 끝나기 때문에 `]`를 바로 이어붙이면 Llama 3.2 1B와 Gemma 3 1B-pt "
    "토크나이저 모두 ` [` + `]`를 ` []`라는 **단일 병합 토큰**으로 재토큰화해서, 토큰 수가 늘지 않고 "
    "assert가 실패한다 (두 토크나이저 모두 `\"[]\"`가 단일 vocab 항목으로 존재)."
)
lines.append("")
lines.append(
    "대안: `prefix + dep + \"]\"` (예: `...}, [page]`)로 닫힌 1-원소 배열 문맥을 만들어 +2 토큰 "
    "assert로 검증하고, 마지막 토큰을 `close_id`로 취했다. 식별자가 대괄호 사이에 있을 때는 `]`가 "
    "항상 독립 토큰으로 분리되며 (`\"[dep]\"` → `[\'[\', \'dep\', \']\']`), 이렇게 얻은 id가 "
    "standalone `\"]\"` 토큰화 결과와 정확히 일치함을 확인했다. 원안의 \"하드코딩하지 말고 문맥에서 "
    "계산\" 원칙은 유지하면서 빈 대괄호 병합 아티팩트만 회피한 것이다."
)
lines.append("")
lines.append("## 1. 전체 토큰 검증 (Stage 1)")
lines.append("")
lines.append(
    "45 sample × 5 condition × {clean, corrupted} = 450 조합에 대해 위 방식으로 "
    "dep_id/alt_id/close_id 계산 및 길이 assert를 수행했다."
)
lines.append("")
lines.append("| model | 통과 |")
lines.append("|---|---|")
lines.append("| Llama 3.2 1B | 450/450 |")
lines.append("| Gemma 3 1B-pt | 450/450 |")
lines.append("")
lines.append("## 2. 조건별 요약 (전체 5조건)")
lines.append("")
lines.append(
    "`LD_copy = logit(dep) - logit(alt)`, `LD_inclusion = logit(dep) - logit(])`. 모든 값은 45 "
    "sample에 대한 median이다. `frac_gap_*_pos`는 `LD_*_clean > LD_*_corrupted`가 성립하는 sample "
    "비율이다."
)
lines.append("")
for model_key in MODEL_KEYS:
    lines.append(f"### {MODEL_LABELS[model_key]}")
    lines.append("")
    lines.append(
        "| condition | LD_inclusion clean | LD_inclusion corrupted | gap_inclusion | "
        "frac gap_inclusion>0 | LD_copy clean | LD_copy corrupted | gap_copy | frac gap_copy>0 | "
        "copy_error corrupted (median / mean) |"
    )
    lines.append("|---|---|---|---|---|---|---|---|---|---|")
    for cond in CONDITIONS:
        row = summary[(summary["model"] == model_key) & (summary["condition"] == cond)].iloc[0]
        lines.append(
            f"| {cond} | {fmt(row['median_LD_inclusion_clean'])} | "
            f"{fmt(row['median_LD_inclusion_corrupted'])} | {fmt(row['median_gap_inclusion'])} | "
            f"{row['frac_gap_inclusion_pos']:.3f} | {fmt(row['median_LD_copy_clean'])} | "
            f"{fmt(row['median_LD_copy_corrupted'])} | {fmt(row['median_gap_copy'])} | "
            f"{row['frac_gap_copy_pos']:.3f} | {fmt(row['median_copy_error_corrupted'])} / "
            f"{fmt(row['mean_copy_error_corrupted'])} |"
        )
    lines.append("")

lines.append("## 3. §12.1 기준 체크리스트 — `useEffect` 조건 중심")
lines.append("")
lines.append(
    "아래는 §12.1에서 `LD_inclusion = logit(dep) - logit(])`을 메인 metric으로 채택하기 위한 5개 "
    "기준을 `useEffect` 조건(n=45)에 대해 그대로 평가한 표다. 기준 1~4는 §12.1에 명시된 부등식이므로 "
    "통과 여부를 그대로 표시했다. 기준 5(`copy_error`가 과하게 양수로 치우치지 않음)는 고정 임계값이 "
    "문서에 없으므로 **수치만 제시**하고 판정은 하지 않았다. 메인 metric 최종 선택은 연구자가 이 표를 "
    "보고 결정한다."
)
lines.append("")
lines.append("| # | 기준 | Llama 3.2 1B | 판정 | Gemma 3 1B-pt | 판정 |")
lines.append("|---|---|---|---|---|---|")
for _, r in checklist.iterrows():
    llama_val = r["llama"] if isinstance(r["llama"], str) else fmt(r["llama"])
    gemma_val = r["gemma"] if isinstance(r["gemma"], str) else fmt(r["gemma"])
    lines.append(f"| {r['#']} | {r['criterion']} | {llama_val} | {r['llama_verdict']} | {gemma_val} | {r['gemma_verdict']} |")
lines.append("")

ll = wide[(wide["model"] == "llama3.2-1b") & (wide["condition"] == "useEffect")]
gg = wide[(wide["model"] == "gemma3-1b-pt") & (wide["condition"] == "useEffect")]

lines.append("### 기준 2 보강: sample 단위 분포 (`useEffect`, n=45)")
lines.append("")
lines.append(
    "median이 아니라 sample 전체에서 LD_inclusion_corrupted < 0가 성립하는 비율도 함께 보고한다 "
    "(median만으로는 분포의 일관성을 알 수 없기 때문)."
)
lines.append("")
lines.append("| model | LD_inclusion_corrupted < 0 인 sample 비율 | min | max |")
lines.append("|---|---|---|---|")
lines.append(
    f"| Llama 3.2 1B | {(ll['LD_inclusion_corrupted']<0).mean():.3f} | "
    f"{fmt(ll['LD_inclusion_corrupted'].min())} | {fmt(ll['LD_inclusion_corrupted'].max())} |"
)
lines.append(
    f"| Gemma 3 1B-pt | {(gg['LD_inclusion_corrupted']<0).mean():.3f} | "
    f"{fmt(gg['LD_inclusion_corrupted'].min())} | {fmt(gg['LD_inclusion_corrupted'].max())} |"
)
lines.append("")

lines.append("## 4. 참고: LD_copy(variable tracking) 분포 — `useEffect`, n=45")
lines.append("")
lines.append("§12.2 판단에 참고할 수 있도록 동일한 방식으로 LD_copy 쪽 sample-level 분포도 함께 보고한다.")
lines.append("")
lines.append("| model | LD_copy_corrupted < 0 인 sample 비율 | min | max |")
lines.append("|---|---|---|---|")
lines.append(
    f"| Llama 3.2 1B | {(ll['LD_copy_corrupted']<0).mean():.3f} | {fmt(ll['LD_copy_corrupted'].min())} | "
    f"{fmt(ll['LD_copy_corrupted'].max())} |"
)
lines.append(
    f"| Gemma 3 1B-pt | {(gg['LD_copy_corrupted']<0).mean():.3f} | {fmt(gg['LD_copy_corrupted'].min())} | "
    f"{fmt(gg['LD_copy_corrupted'].max())} |"
)
lines.append("")

lines.append("## 5. 출력 파일")
lines.append("")
lines.append(
    "- `baseline_logits_3tokens.csv`: raw long format (model, condition, sample_id, run_type 별 "
    "logit(dep)/logit(alt)/logit(]) 및 파생 지표). 900 row."
)
lines.append(
    "- `baseline_wide_per_sample.csv`: 위 raw를 (model, condition, sample_id) 단위로 clean/corrupted를 "
    "나란히 두고 gap_copy/gap_inclusion을 계산한 중간 산출물."
)
lines.append("- `baseline_summary_table.csv`: 본 문서 §2 표의 원본 수치.")
lines.append("")
lines.append("## 6. 메인 metric 선택은 보류")
lines.append("")
lines.append(
    "본 문서는 §12 기준의 통과 여부와 수치만 제시한다. `LD_inclusion`을 메인으로 할지 `LD_copy`를 "
    "메인으로 할지, 혹은 §12.3의 mixed-result 처리를 적용할지는 연구자가 위 결과를 보고 결정한다."
)
lines.append("")

with open("baseline_summary.md", "w") as f:
    f.write("\n".join(lines))

print("wrote baseline_summary.md")

wrote baseline_summary.md


## 결론

- 전체 dataset(45 sample × 5 condition × clean/corrupted)에서 token validation 450/450 통과 (양 모델).
- `useEffect` 조건에서 §12.1 기준 1, 3, 4는 PASS, 기준 2는 두 모델 모두 FAIL (corrupted prefix에서도
  45/45 sample 전부 `logit(dep) > logit(])`가 유지됨 — empty-array 회피 prior가 강하게 작용하는 것으로
  보인다).
- 같은 조건에서 `LD_copy`(variable tracking)는 거의 모든 sample에서 clean/corrupted 부호가 뒤집힌다
  (Llama 41/45, Gemma 45/45).
- 최종 메인 metric 선택은 이 결과를 검토한 연구자가 결정한다.